# MoE架构
-  **专家网络**：系统包含多个独立的神经网络（可以是简单的前馈网络）。每个专家不再需要处理全局任务，只需专注于输入空间中的一个局部区域或一类特定的子任务。
-  **门控网络**：充当协调者的角色。它接收与专家相同的输入 $x$，并输出一组**混合比例（Mixing Proportions）**$p_i$，即选择每个专家的概率。它就像一个软性的随机开关，决定当前的输入案例应该由哪位专家来主导处理。</li>
设计了特定的机制来鼓励专家之间的**竞争（Competition）**，而非合作。如果仅仅将所有专家的输出简单线性相加来逼近目标 $\mathbf{y}$，即预测值 $\hat{\mathbf{y}} = \sum p_i \mathbf{E}_i$，并在最终输出上计算误差，专家们会倾向于“合作”。

$$ Loss_{coop} = || \mathbf{y} - \sum_{i} p_i \mathbf{E}_i ||^2 \tag{6.2} $$

在这种合作模式下，为了减小总误差，每个专家都会试图去弥补其他专家的残差。继而导致每个案例都牵动所有专家的权重，依然无法解决“干扰效应”。为了实现解耦，作者提出将系统视为一个随机生成器，并采用**负对数似然**作为目标函数：

$$ Loss_{comp} = - \log \sum_{i} p_i e^{-\frac{1}{2} || \mathbf{y} - \mathbf{E}_i ||^2} \tag{6.3} $$

其中：
-   $\mathbf{y}$ 是我们希望模型输出的**真实目标**。
-   $\mathbf{E}_i$ 是第 $i$ 个**专家**的输出。
-   $p_i$ 是门控网络分配给第 $i$ 个专家的**权重**（概率）。

在这个目标函数中，系统倾向于**“赢家通吃”**。当某个专家 $\mathbf{E}_i$ 的输出非常接近目标 $\mathbf{y}$ 时（即误差项 $|| \mathbf{y} - \mathbf{E}_i ||^2$ 很小），指数项 $e^{-\dots}$ 会变得很大。为了最小化总 Loss，门控网络会倾向于显著增加这个“表现好”的专家的权重 $p_i$，而忽略其他专家。这一机制实现了**权重的解耦**。误差反向传播时，只有被选中的“胜出者”和门控网络的权重会被更新，其他专家几乎不受影响。**有效缓解了**任务间的干扰，实现了“让专业的人干专业的事”。

-   **保留 Attention**：Transformer 的 Self-Attention 层保持不变，因为其参数量相对较小且计算关键。
-   **替换 FFN**：将 Transformer Block 中的**前馈神经网络**替换为 MoE 层。
-   **隔层设置**：通常采用“隔层替换”的策略（例如第 1、3、5 层使用 MoE，第 2、4、6 层保留标准 FFN），在增加容量和保持稳定性之间取得平衡。

对于 MoE 层的计算，GShard 明确了输入 Token $\mathbf{x}$ 的输出 $\mathbf{y}$ 是由门控网络 $\mathcal{G}$ 选择的专家输出的加权和：

$$ \mathbf{y} = \sum_{i=1}^{N} p_i(\mathbf{x}) \cdot \mathbf{E}_i(\mathbf{x}) \tag{6.4} $$

其中：
-   $p_i(\mathbf{x})$ 是门控网络（Router）计算出的第 $i$ 个专家的**权重**（通常是 Softmax 后的 Top-k 概率，其余为 0）。
-   $\mathbf{E}_i(\mathbf{x})$ 是第 $i$ 个**专家网络**（Expert FFN）对输入 $\mathbf{x}$ 的处理结果。

每个专家 $\mathbf{E}_i$ 内部通常就是一个标准的双层全连接网络：

$$ \mathbf{E}_i(\mathbf{x}) = \mathbf{W}_{out} \cdot \text{ReLU}(\mathbf{W}_{in} \cdot \mathbf{x}) \tag{6.5} $$

# Deepseek
使用共享专家策略，避免知识冗余
$$ \mathbf{y} = \underbrace{\sum_{i \in \mathcal{S}} \mathbf{E}i(\mathbf{x})}{\text{Shared Experts}} + \underbrace{\sum_{j \in \text{TopK}(\mathcal{R})} p_j(\mathbf{x}) \cdot \mathbf{E}j(\mathbf{x})}{\text{Routed Experts}} \tag{6.8} $$


In [ ]:
import torch.nn as nn
import torch

class FeedForward(nn.Module):
    def __init__(self,dim:int,hidden_dim:int,multi_of:int):
        super().__init__()
        hidden_dim = int(hidden_dim * 2/3)
        hidden_dim = multi_of * ((hidden_dim + multi_of -1)//multi_of)
        self.w1 = nn.Linear(dim,hidden_dim)
        self.w2 = nn.Linear(hidden_dim,dim)
        self.w3 = nn.Linear(dim,hidden_dim)
    def forward(self,x:torch.Tensor):
        return self.w2(torch.nn.functional.silu(self.w1(x)) * self.w3(x))
    
    
class MoE(nn.Module):
    def __init__(self,dim:int,hidden_dim:int,multi_of:int=4,num_experts:int=8,top_k:int=2):
        super().__init__()
        self.top_k = top_k
        self.num_experts = num_experts
        self.experts = nn.ModuleList([FeedForward(dim,hidden_dim,multi_of) for _ in range(num_experts)])
        self.gate = nn.Linear(dim,num_experts)
    def forward(self,x:torch.Tensor):
        B,T,D = x.shape #(batch,seq_len,dim)
        x_flatten = x.reshape(-1,D) #(B*T,D)
        outputs = torch.zeros_like(x_flatten) #(B*T,D)
        gate_logits = self.gate(x_flatten) #(B*T,num_experts)
        weights,indices = torch.topk(gate_logits,self.top_k,dim=-1) #(B*T,top_k)
        weights = torch.nn.functional.softmax(weights,dim=-1) #(B*T,top_k)
        for i,expert in enumerate(self.experts):
            batch_id,k_id = (indices==i)
            if batch_id.sum() ==0:
                continue
            expert_input = x_flatten[batch_id] #(num_selected,D)
            expert_output = expert(expert_input) #(num_selected,D)
            weight = weights[batch_id,k_id].unsqueeze(-1) #(num_selected,1)
            outputs.index_add_(0,batch_id,expert_output * weight) #(B*T,D)
        return outputs.reshape(B,T,D)
    

In [ ]:
import torch
import torch.nn as nn
class RMSNorm(nn.Module):
    def __init__(self,dim: int, eps:float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        norm_x = x / torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm_x * self.weight
    

def precompute_freqs_cis(dim:int,end:int,theta:float=10000):
    freqs = 1.0/(theta**(torch.arange(0,dim,2)/dim))
    pos = torch.arange(0,end)
    freqs = torch.outer(pos,freqs) ## (end,dim/2)算角度
    freqs_cis = torch.polar(torch.ones_like(freqs),freqs) ## (end,dim/2),复数表示
    return freqs_cis##end=

def reshape_for_broadcast(x:torch.Tensor,freqs_cis:torch.Tensor)->torch.Tensor:
    ndim = x.dim() #(batch,seq_len,num_heads,head_dim/2)
    shape = [1] * ndim
    shape[1] = x.shape[1]
    shape[-1] = x.shape[-1]
    freqs_cis = freqs_cis[:x.shape[1]]  # (actual_seq_len, D//2)
    return freqs_cis.view(*shape)


def apply_rotary_emb(x_q:torch.Tensor,x_k:torch.Tensor,freqs_cis:torch.Tensor):
    xq_ = torch.view_as_complex(x_q.float().reshape(*x_q.shape[:-1],-1,2)) #(batch,seq_len,num_heads,head_dim/2),复数形式
    xk_ = torch.view_as_complex(x_k.float().reshape(*x_k.shape[:-1],-1,2))
    freqs_q = reshape_for_broadcast(xq_,freqs_cis)
    freqs_k = reshape_for_broadcast(xk_,freqs_cis)
    xq_out = torch.view_as_real(xq_*freqs_q).flatten(-2) #(batch,seq_len,num_heads,head_dim)
    xk_out = torch.view_as_real(xk_*freqs_k).flatten(-2)
    return xq_out.type_as(x_q),xk_out.type_as(x_k)

import torch.nn as nn
def repeat_kv(x:torch.Tensor,n_rep:int) -> torch.Tensor:
    batch_size,seq_len,num_heads,head_dim = x.shape
    if n_rep == 1:
        return x
    return(x[:,:,:,None,:].expand(batch_size,seq_len,num_heads,n_rep,head_dim).reshape(batch_size,seq_len,num_heads*n_rep,head_dim))

class GroupedQueryAttention(nn.Module):
    def __init__(self,dim:int,n_heads:int,n_kv_heads:int):
        super().__init__()
        self.n_heads = n_heads
        self.dim = dim
        self.head_dim = dim // n_heads
        self.n_kv_heads = n_kv_heads
        self.rep = n_heads // n_kv_heads
        self.to_q = nn.Linear(dim,n_heads * self.head_dim,bias=False)
        self.to_k = nn.Linear(dim,n_kv_heads * self.head_dim,bias=False)
        self.to_v = nn.Linear(dim,n_kv_heads * self.head_dim,bias=False)
        self.fc = nn.Linear(n_heads * self.head_dim,dim)
    def forward(self,x:torch.Tensor,freqs_cis:torch.Tensor,mask):
        q = self.to_q(x).view(x.shape[0],x.shape[1],self.n_heads,self.head_dim)#(batch,seq_len,n_heads,head_dim)
        k = self.to_k(x).view(x.shape[0],x.shape[1],self.n_kv_heads,self.head_dim)#(batch,seq_len,n_kv_heads,head_dim)
        v = self.to_v(x).view(x.shape[0],x.shape[1],self.n_kv_heads,self.head_dim)
        q,k = apply_rotary_emb(q,k,freqs_cis)
        k = repeat_kv(k,self.rep)
        v = repeat_kv(v,self.rep)
        scores = torch.matmul(q.transpose(1,2),k.transpose(1,2).transpose(-2,-1)) / (self.head_dim ** 0.5) #(batch,n_heads,seq_len,seq_len)
        if mask is not None:
            scores = scores+mask
        attention = torch.softmax(scores,dim=-1)
        out = torch.matmul(attention,v.transpose(1,2)) #(batch,n_heads,seq_len,head_dim)
        out = out.transpose(1,2).contiguous().view(x.shape[0],x.shape[1],self.n_heads * self.head_dim)
        out = self.fc(out)
        return out

class FeedForward(nn.Module):
    def __init__(self,dim:int,hidden_dim:int,multi_of:int):
        super().__init__()
        hidden_dim = int(hidden_dim * 2/3)
        hidden_dim = multi_of * ((hidden_dim + multi_of -1)//multi_of)
        self.w1 = nn.Linear(dim,hidden_dim)
        self.w2 = nn.Linear(hidden_dim,dim)
        self.w3 = nn.Linear(dim,hidden_dim)
    def forward(self,x:torch.Tensor):
        return self.w2(torch.nn.functional.silu(self.w1(x)) * self.w3(x))
    
class MoE(nn.Module):
    def __init__(self,dim:int,ffn_hidden_multplier:float,multi_of:int=4,num_experts:int=8,top_k:int=2):
        super().__init__()
        self.top_k = top_k
        self.num_experts = num_experts
        hidden_dim = dim * 4
        self.experts = nn.ModuleList([FeedForward(dim,hidden_dim,multi_of) for _ in range(num_experts)])
        self.gate = nn.Linear(dim,num_experts)
    def forward(self,x:torch.Tensor):
        B,T,D = x.shape #(batch,seq_len,dim)
        x_flatten = x.reshape(-1,D) #(B*T,D)
        outputs = torch.zeros_like(x_flatten) #(B*T,D)
        gate_logits = self.gate(x_flatten) #(B*T,num_experts)
        weights,indices = torch.topk(gate_logits,self.top_k,dim=-1) #(B*T,top_k)
        weights = torch.nn.functional.softmax(weights,dim=-1) #(B*T,top_k)
        for i,expert in enumerate(self.experts):
            batch_id,k_id = torch.where(indices==i)
            if len(batch_id) ==0:
                continue
            expert_input = x_flatten[batch_id] #(num_selected,D)
            expert_output = expert(expert_input) #(num_selected,D)
            weight = weights[batch_id,k_id].unsqueeze(-1) #(num_selected,1)
            outputs.index_add_(0,batch_id,expert_output * weight) #(B*T,D)
        return outputs.reshape(B,T,D)
    
class TransformerBlock(nn.Module):
    def __init__(self,dim:int,n_heads:int,n_kv_heads:int,multi_of:int,ffn_hidden_multplier:float=None,eps:float=1e-6):
        super().__init__()
        self.attention = GroupedQueryAttention(dim,n_heads,n_kv_heads)
        self.norm1 = RMSNorm(dim)
        self.ff = MoE(dim,ffn_hidden_multplier,multi_of,num_experts=8,top_k=2)
        self.norm2 = RMSNorm(dim)
    def forward(self,x:torch.Tensor,start_pos: int,freqs_cis:torch.Tensor,mask):
        x = x + self.attention(self.norm1(x),freqs_cis,mask)
        x = x + self.ff(self.norm2(x))
        return x
class LlamaModel(nn.Module):
    def __init__(self,vocab_size:int,dim:int,n_heads:int,n_kv_heads:int,n_layers:int,max_seq_len:int,multi_of:int,ffn_dim_multiplier:float=None,norm_eps:float=1e-6,max_batch_size:int=4):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size,dim)
        self.layers = nn.ModuleList([TransformerBlock(dim,n_heads,n_kv_heads,multi_of) for _ in range(n_layers)])
        self.norm = RMSNorm(dim)
        self.output = nn.Linear(dim,vocab_size,bias=False)
        self.register_buffer('freqs_cis',precompute_freqs_cis(dim//n_heads,max_seq_len))
    def forward(self,input:torch.Tensor,start_pos:int=0):
        x = self.token_emb(input)
        freqs_cis = self.freqs_cis[start_pos:start_pos + x.shape[1]]
        mask = None
        seq_len = x.shape[1]
        if seq_len > 1:
            mask = torch.full((seq_len,seq_len),float('-inf'))
            mask = torch.triu(mask,diagonal=1).to(x.device)
            mask = torch.hstack([torch.zeros((seq_len,start_pos),device=x.device),mask])
        for layer in self.layers:
            x = layer(x,start_pos,freqs_cis,mask)
        x = self.norm(x)
        x = self.output(x)
        return x
    
def main() -> None:
    # 使用小尺寸参数，便于 CPU/GPU 都能快速跑通
    model = LlamaModel(
        vocab_size=1000,
        dim=256,
        n_layers=2,
        n_heads=8,
        n_kv_heads=2,
        multi_of=64,
        ffn_dim_multiplier=None,
        norm_eps=1e-6,
        max_batch_size=4,
        max_seq_len=64,
    )

    # 构造随机 token 序列并执行前向
    batch_size, seq_len = 2, 16
    tokens = torch.randint(0, 1000, (batch_size, seq_len))
    logits = model(tokens, start_pos=0)

    # 期望: [batch_size, seq_len, vocab_size]
    print("logits shape:", tuple(logits.shape))

if __name__ == "__main__":
    main()

logits shape: (2, 16, 1000)


: 